# Creating the Dataset

In [1]:
!pip install pandas nltk

Extracting comments from the repositories

In [2]:
!git clone https://github.com/pallets/flask.git
!git clone https://github.com/psf/requests.git

Cloning into 'flask'...
remote: Enumerating objects: 26139, done.
remote: Counting objects: 100% (275/275), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 26139 (delta 218), reused 148 (delta 147), pack-reused 25864 (from 4)
Receiving objects: 100% (26139/26139), 11.55 MiB | 15.70 MiB/s, done.
Resolving deltas: 100% (17444/17444), done.
Cloning into 'requests'...
remote: Enumerating objects: 26455, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 26455 (delta 101), reused 42 (delta 42), pack-reused 26303 (from 3)
Receiving objects: 100% (26455/26455), 13.02 MiB | 21.72 MiB/s, done.
Resolving deltas: 100% (17331/17331), done.


In [4]:
!git clone https://github.com/pandas-dev/pandas.git

Cloning into 'pandas'...
remote: Enumerating objects: 425073, done.
remote: Counting objects: 100% (968/968), done.
remote: Compressing objects: 100% (591/591), done.
remote: Total 425073 (delta 702), reused 377 (delta 377), pack-reused 424105 (from 5)
Receiving objects: 100% (425073/425073), 378.38 MiB | 21.94 MiB/s, done.
Resolving deltas: 100% (357659/357659), done.


In [5]:
import os

python_files = []

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith(".py"):
            python_files.append(os.path.join(root, file))

print("Total Python files found:", len(python_files))

Total Python files found: 1625


In [6]:
import random

# take only a sample of files
python_files = random.sample(python_files, 250)

print("Files selected for analysis:", len(python_files))

Files selected for analysis: 250


In [10]:
import tokenize

def extract_comments(file_path):

    comments = []

    try:
        with open(file_path, "r", encoding="utf-8") as f:

            tokens = tokenize.generate_tokens(f.readline)

            for toknum, tokval, _, _, _ in tokens:
                if toknum == tokenize.COMMENT:
                    comments.append(tokval)

    except:
        pass

    return comments

In [11]:
all_comments = []

for file in python_files:

    comments = extract_comments(file)

    for comment in comments:
        all_comments.append({
            "file": file,
            "comment": comment
        })

print("Total comments extracted:", len(all_comments))

Total comments extracted: 6426


In [12]:
import pandas as pd

df = pd.DataFrame(all_comments)

df.head()

,file,comment
0,/content/pandas/pandas/io/sas/sas_constants.py,"# Keep ""page_comp_type"" bits"
1,/content/pandas/pandas/io/sas/sas_constants.py,"# Incomplete list of encodings, using SAS nome..."
2,/content/pandas/pandas/io/sas/sas_constants.py,# https://support.sas.com/documentation/online...
3,/content/pandas/pandas/io/sas/sas_constants.py,# corresponding to the Python documentation of...
4,/content/pandas/pandas/io/sas/sas_constants.py,# https://docs.python.org/3/library/codecs.htm...


In [13]:
df.to_csv("comments_dataset.csv", index=False)

print("Dataset saved successfully!")

Dataset saved successfully!


In [14]:
df = df.sample(2000, random_state=42)

print("Sampled dataset size:", df.shape)

Sampled dataset size: (2000, 2)


In [15]:
import re

def clean_comment(text):

    text = text.lower()
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)

    return text.strip()

df["clean_comment"] = df["comment"].apply(clean_comment)

df.head()

,file,comment,clean_comment
324,/content/pandas/pandas/core/internals/construc...,# and a subsequent `astype` will not already r...,and a subsequent astype will not already resul...
5396,/content/pandas/pandas/tests/util/test_assert_...,# Do not ignore index.,do not ignore index
4894,/content/pandas/pandas/tests/tools/test_to_dat...,# Match Timestamp behavior in disallowing non-...,match timestamp behavior in disallowing nonrou...
2548,/content/pandas/pandas/tests/arrays/sparse/tes...,# x: ----- -----,x
381,/content/pandas/pandas/core/internals/construc...,# TODO: is that an issue with numpy?,todo is that an issue with numpy


In [16]:
df = df[df["clean_comment"] != ""]

In [17]:
df.to_csv("comments_dataset.csv", index=False)

print("Dataset saved successfully")

Dataset saved successfully


In [19]:
df.shape

(1958, 3)

In [20]:
df = df[df["clean_comment"].str.len() > 5]

print("Dataset size after removing short comments:", df.shape)

Dataset size after removing short comments: (1619, 3)


In [21]:
license_words = ["copyright", "license", "apache", "mit"]

df = df[~df["clean_comment"].str.contains('|'.join(license_words))]

print("Dataset size after removing license comments:", df.shape)

Dataset size after removing license comments: (1617, 3)


In [22]:
df = df[~df["clean_comment"].str.match(r'^[^a-zA-Z]+$')]

In [23]:
df.head(10)

,file,comment,clean_comment
324,/content/pandas/pandas/core/internals/construc...,# and a subsequent `astype` will not already r...,and a subsequent astype will not already resul...
5396,/content/pandas/pandas/tests/util/test_assert_...,# Do not ignore index.,do not ignore index
4894,/content/pandas/pandas/tests/tools/test_to_dat...,# Match Timestamp behavior in disallowing non-...,match timestamp behavior in disallowing nonrou...
381,/content/pandas/pandas/core/internals/construc...,# TODO: is that an issue with numpy?,todo is that an issue with numpy
3551,/content/pandas/pandas/core/arrays/datetimelik...,# All of the functions implemented here are or...,all of the functions implemented here are ordi...
4991,/content/pandas/pandas/tests/tools/test_to_dat...,# CASE 1: valid input,case valid input
1345,/content/pandas/pandas/tests/strings/conftest.py,"# if the string is not found, search only for ...",if the string is not found search only for emp...
1807,/content/pandas/pandas/tests/series/methods/te...,"# float, int, datetime64 (use i8), timedelts64...",float int datetime use i timedelts same
1721,/content/pandas/pandas/tests/frame/test_query_...,# Note: Formatting checks may wrongly consider...,note formatting checks may wrongly consider th...
3807,/content/pandas/pandas/core/computation/scope.py,# these are created when parsing indexing expr...,these are created when parsing indexing expres...


In [24]:
df.to_csv("comments_dataset_cleaned.csv", index=False)

print("Clean dataset saved!")

Clean dataset saved!


# Author : Venkatesh Paitwar